In [1]:
import numpy as np
import tensorflow as tf
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences
from tensorflow.keras.models import Model
from tensorflow.keras.layers import Input, Embedding, SimpleRNN, Dense

In [2]:
sentences = [
 "I love this product",
 "This movie made me smile",
 "Service was friendly and quick",
 "Today felt bright and happy",
 "This is the best day",
 "Absolutely fantastic experience",
 "I enjoyed every single moment",
 "Great job, well done",
 "The food tasted delicious",
 "Totally recommend to everyone",
 "Very satisfied with results",
 "This worked better than expected",
 "Amazing quality and value",
 "Such a pleasant surprise",
 "I feel positive about this",
 "I hate this product",
 "This movie bored me",
 "Service was rude and slow",
 "Today was cold and lonely",
 "This is the worst day",
 "Terrible experience overall",
 "I regret buying this",
 "Very disappointed with results",
 "The food tasted awful",
 "Do not recommend this",
 "It broke after one use",
 "Not worth the money",
 "Utterly frustrating and annoying",
 "I feel negative about this",
 "Such a waste of time",
]
labels = [1]*15 + [0]*15
labels = np.array(labels)

In [3]:
vocab_size = 200

toke = Tokenizer(num_words=vocab_size, oov_token = "<OOV>")
toke.fit_on_texts(sentences)

In [4]:
seqs = toke.texts_to_sequences(sentences)
max_len = max(len(s) for s in seqs)
X = pad_sequences(seqs, maxlen = max_len, padding='post')
y = labels


In [5]:
embed_dim = 16
rnn_units = 8

In [10]:
inp = Input(shape=(max_len,), dtype="int32", name="input")
x = Embedding(vocab_size, embed_dim, mask_zero=True, name = "embed")(inp)
# When return_sequences=True and return_state=True, SimpleRNN returns a tuple:
# (output_sequence, last_hidden_state)
output_sequence, last_hidden_state = SimpleRNN(
    units=rnn_units, return_sequences=True, return_state=True, name="simple_rnn"
)(x)

# For binary classification, we typically use the last hidden state
x_last = last_hidden_state
out = Dense(1, activation="sigmoid", name="out")(x_last)
model = Model(inputs=inp, outputs=out)
model.compile(loss =  "binary_crossentropy", optimizer="adam", metrics=['accuracy'])
model.summary()

Model: "functional"

┏━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┓
┃ Layer (type)        ┃ Output Shape      ┃    Param # ┃ Connected to      ┃
┡━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━┩
│ input (InputLayer)  │ (None, 5)         │          0 │ -                 │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ embed (Embedding)   │ (None, 5, 16)     │      3,200 │ input[0][0]       │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ not_equal_3         │ (None, 5)         │          0 │ input[0][0]       │
│ (NotEqual)          │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ simple_rnn          │ [(None, 5, 8),    │        200 │ embed[0][0],      │
│ (SimpleRNN)         │ (None, 8)]        │            │ not_equal_3[0][0] │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ out (Dense)         │ (None, 1)         │          9 │ simple_rnn[0][1]  │
└─────────────────────┴───────────────────┴────────────┴───────────────────┘

 Total params: 3,409 (13.32 KB)

 Trainable params: 3,409 (13.32 KB)

 Non-trainable params: 0 (0.00 B)

In [11]:
model.fit(X, y, epochs=30, batch_size = 8, verbose = 1)

Epoch 1/30
4/4 ━━━━━━━━━━━━━━━━━━━━ 2s 13ms/step - accuracy: 0.5892 - loss: 0.6947
Epoch 2/30
4/4 ━━━━━━━━━━━━━━━━━━━━ 0s 13ms/step - accuracy: 0.6792 - loss: 0.6848
Epoch 3/30
4/4 ━━━━━━━━━━━━━━━━━━━━ 0s 12ms/step - accuracy: 0.6500 - loss: 0.6774
Epoch 4/30
4/4 ━━━━━━━━━━━━━━━━━━━━ 0s 11ms/step - accuracy: 0.6892 - loss: 0.6623
Epoch 5/30
4/4 ━━━━━━━━━━━━━━━━━━━━ 0s 12ms/step - accuracy: 0.7067 - loss: 0.6518
Epoch 6/30
4/4 ━━━━━━━━━━━━━━━━━━━━ 0s 15ms/step - accuracy: 0.7700 - loss: 0.6449
Epoch 7/30
4/4 ━━━━━━━━━━━━━━━━━━━━ 0s 11ms/step - accuracy: 0.7908 - loss: 0.6235
Epoch 8/30
4/4 ━━━━━━━━━━━━━━━━━━━━ 0s 11ms/step - accuracy: 0.8850 - loss: 0.6017
Epoch 9/30
4/4 ━━━━━━━━━━━━━━━━━━━━ 0s 12ms/step - accuracy: 0.9067 - loss: 0.5935
Epoch 10/30
4/4 ━━━━━━━━━━━━━━━━━━━━ 0s 11ms/step - accuracy: 0.9442 - loss: 0.5845
Epoch 11/30
4/4 ━━━━━━━━━━━━━━━━━━━━ 0s 11ms/step - accuracy: 0.9525 - loss: 0.5636
Epoch 12/30
4/4 ━━━━━━━━━━━━━━━━━━━━ 0s 11ms/step - accuracy: 0.9658 - loss: 0.5554
E

In [12]:
intermediate_model = Model(inputs = model.inputs, outputs = model.get_layer("simple_rnn").output)
intermediate_output = intermediate_model.predict(X)
intermediate_output

1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 351ms/step


[array([[[-0.05749067,  0.01809803, -0.02918015, ...,  0.02649966,
           0.04542303,  0.07583054],
         [ 0.02484168,  0.19258094,  0.2081003 , ..., -0.05573189,
          -0.18572675, -0.31718066],
         [ 0.49263826,  0.19443011,  0.1719774 , ..., -0.21055359,
          -0.23855403,  0.03656404],
         [ 0.27250105,  0.2566114 , -0.34154716, ...,  0.33023742,
          -0.5625658 ,  0.26510257],
         [ 0.27250105,  0.2566114 , -0.34154716, ...,  0.33023742,
          -0.5625658 ,  0.26510257]],
 
        [[ 0.03613638,  0.01349147,  0.02937288, ..., -0.03168384,
           0.09482642,  0.03853637],
         [ 0.04498384,  0.01808991, -0.17232299, ...,  0.10098845,
           0.16207795,  0.02487355],
         [-0.23978968, -0.02810124,  0.26067662, ..., -0.27984762,
          -0.17198604, -0.4520852 ],
         [ 0.26322815,  0.3004227 ,  0.47105753, ..., -0.54576975,
          -0.36069426,  0.15464681],
         [ 0.3827175 ,  0.5650966 , -0.64560115, ...,  0.3511

In [13]:
test_sentences = [
    "This is a wonderful product!",
    "I absolutely hate this movie.",
    "The service was average.",
    "Feeling great today.",
    "What a terrible waste of time."
]

# Tokenize the test sentences
test_seqs = toke.texts_to_sequences(test_sentences)

# Pad the test sequences to the same max_len
X_test = pad_sequences(test_seqs, maxlen=max_len, padding='post')

# Make predictions
predictions = model.predict(X_test)

# Display predictions along with the original sentences
print("Sentiment Predictions:")
for i, sentence in enumerate(test_sentences):
    sentiment = "Positive" if predictions[i][0] > 0.5 else "Negative"
    print(f"Sentence: '{sentence}' | Prediction: {predictions[i][0]:.4f} ({sentiment})")

1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 179ms/step
Sentiment Predictions:
Sentence: 'This is a wonderful product!' | Prediction: 0.7436 (Positive)
Sentence: 'I absolutely hate this movie.' | Prediction: 0.4203 (Negative)
Sentence: 'The service was average.' | Prediction: 0.5091 (Positive)
Sentence: 'Feeling great today.' | Prediction: 0.5811 (Positive)
Sentence: 'What a terrible waste of time.' | Prediction: 0.0673 (Negative)
